# Fine-tuning XLS-R for Hindi

* **Wav2Vec2** is a pretrained model for Automatic Speech Recognition (ASR)  
* *Facebook AI* presented a multi-lingual version of Wav2Vec2, called XLSR. XLSR stands for *cross-lingual speech representations* and refers to model's ability to learn speech representations that are useful across multiple languages.

* Similar to BERT's masked language modeling objective, XLS-R learns contextualized speech representations by randomly masking feature vectors before passing them to a transformer network during self-supervised pre-training.
* XLS-R is fine-tuned using **Connectionist Temporal Classification (CTC)**, which is an algorithm that is used to train neural networks for sequence-to-sequence problems, such as ASR.
* The Connectionist Temporal Classification (CTC) algorithm works by aligning the input audio features with the output sequence of tokens. It uses a blank token to handle variable-length alignments and predicts the most likely token sequence given the input features, enabling end-to-end training without requiring explicit alignment annotations.


![wav2vec2_structure](https://raw.githubusercontent.com/patrickvonplaten/scientific_images/master/xls_r.png)



## Checking the GPU info

In [ ]:
gpu_info = !nvidia-smi
gpu_info = '\n'.join(gpu_info)
if gpu_info.find('failed') >= 0:
  print('Not connected to a GPU')
else:
  print(gpu_info)

## Hugging face Login (optional)
## Purposes
* Saving the model.
* Sometimes needs for training in colab.
* Evaluation
* Open-sourcing or private(backup)

In [ ]:
# Executing this code asks for access token
# which needs to be created by users in their hugging face account

from huggingface_hub import notebook_login
# REDACTED_CONFIGURE_TOKEN_OUTSIDE_SOURCE
notebook_login()

In [ ]:
%%capture
!pip install datasets
!pip install transformers==4.11.3
!pip install torchaudio==0.10.0+cu113 -f https://download.pytorch.org/whl/cu113/torch_stable.html
!pip install jiwer
!pip install transformers[torch]  # additional requirements for training in colab
!pip install accelerate -U        # additional requirements for training in colab

In [ ]:
# Need this to upload ypur trained checkpoint to hugging face

# %%capture
# !apt install git-lfs

## Prepare Data, Tokenizer, Feature Extractor

## Loading Dataset from hugging face

In [ ]:
from datasets import load_dataset

# Load the Common Voice dataset for Hindi ("hi") with the correct version number
common_voice_train = load_dataset("mozilla-foundation/common_voice_11_0", "hi", split="train")
common_voice_test = load_dataset("mozilla-foundation/common_voice_11_0", "hi", split="test")


Many ASR datasets only provide the target text, `'sentence'` for each audio array `'audio'` and file `'path'`. Common Voice actually provides much more information about each audio file, such as the `'accent'`, etc. Keeping the notebook as general as possible, we only consider the transcribed text for fine-tuning.



In [ ]:
common_voice_train

In [ ]:
common_voice_test

## Removing unwanted columns

In [ ]:
common_voice_train = common_voice_train.remove_columns(["accent", "age", "client_id", "down_votes", "gender", "locale", "segment", "up_votes"])
common_voice_test = common_voice_test.remove_columns(["accent", "age", "client_id", "down_votes", "gender", "locale", "segment", "up_votes"])

## Short function to display some random samples

In [ ]:
from datasets import ClassLabel
import random
import pandas as pd
from IPython.display import display, HTML

def show_random_elements(dataset, num_examples=10):
    assert num_examples <= len(dataset), "Can't pick more elements than there are in the dataset."
    picks = []
    for _ in range(num_examples):
        pick = random.randint(0, len(dataset)-1)
        while pick in picks:
            pick = random.randint(0, len(dataset)-1)
        picks.append(pick)

    df = pd.DataFrame(dataset[picks])
    display(HTML(df.to_html()))

In [ ]:
show_random_elements(common_voice_train.remove_columns(["path", "audio"]), num_examples=10)

## Removing unwanted symbols

In [ ]:
import re
chars_to_remove_regex = '[\,\?\.\!\-\;\:\"\“\%\‘\”\�\']'

def remove_special_characters(batch):
    batch["sentence"] = re.sub(chars_to_remove_regex, '', batch["sentence"]).lower()
    return batch

In [ ]:
common_voice_train = common_voice_train.map(remove_special_characters)
common_voice_test = common_voice_test.map(remove_special_characters)

Let's look at the processed text labels again.

In [ ]:
show_random_elements(common_voice_train.remove_columns(["path","audio"]))

## Creating Vocabulary
* In CTC, it is common to classify speech chunks into letters, so we will do the same here.
* Let's extract all distinct letters of the training and test data and build our vocabulary from this set of letters.

* We write a mapping function that concatenates all transcriptions into one long transcription and then transforms the string into a set of chars.
* **Note**:   It is important to pass the argument `batched=True` to the `map(...)` function so that the mapping function has access to all transcriptions at once.

In [ ]:
def extract_all_chars(batch):
  all_text = " ".join(batch["sentence"])
  vocab = list(set(all_text))
  return {"vocab": [vocab], "all_text": [all_text]}

In [ ]:
vocab_train = common_voice_train.map(extract_all_chars, batched=True, remove_columns=common_voice_train.column_names)
vocab_test = common_voice_test.map(extract_all_chars, batched=True, remove_columns=common_voice_test.column_names)

Now, we create the union of all distinct letters in the training dataset and test dataset and convert the resulting list into an enumerated dictionary.

In [ ]:
vocab_list = list(set(vocab_train["vocab"][0]) | set(vocab_test["vocab"][0]))

In [ ]:
vocab_dict = {v: k for k, v in enumerate(sorted(vocab_list))}
vocab_dict

In [ ]:
vocab_dict["|"] = vocab_dict[" "]
del vocab_dict[" "]

## Adding Padding Tokens
* Finally, we also add a padding token that corresponds to CTC's "*blank token*". The "blank token" is a core component of the CTC algorithm.

In [ ]:
vocab_dict["[UNK]"] = len(vocab_dict)
vocab_dict["[PAD]"] = len(vocab_dict)
len(vocab_dict)

* Saving the vocabulary as a json file.

In [ ]:
import json
with open('vocab.json', 'w') as vocab_file:
    json.dump(vocab_dict, vocab_file)

* In a final step, we use the json file to load the vocabulary into an instance of the `Wav2Vec2CTCTokenizer` class.

In [ ]:
from transformers import Wav2Vec2CTCTokenizer

tokenizer = Wav2Vec2CTCTokenizer.from_pretrained("./", unk_token="[UNK]", pad_token="[PAD]", word_delimiter_token="|")

## To upload the tokenizer to the hub run the below cell

In [ ]:
# repo_name = "your repo name"
# tokenizer.push_to_hub(repo_name)

## Creating `Wav2Vec2FeatureExtractor`

A `Wav2Vec2FeatureExtractor` object requires the following parameters to be instantiated:

- `feature_size`: Speech models take a sequence of feature vectors as an input. While the length of this sequence obviously varies, the feature size should not. In the case of Wav2Vec2, the feature size is 1 because the model was trained on the raw speech signal ${}^2$.
- `sampling_rate`: The sampling rate at which the model is trained on.
- `padding_value`: For batched inference, shorter inputs need to be padded with a specific value
- `do_normalize`: Whether the input should be *zero-mean-unit-variance* normalized or not. Usually, speech models perform better when normalizing the input
- `return_attention_mask`: Whether the model should make use of an `attention_mask` for batched inference. In general, XLS-R models checkpoints should **always** use the `attention_mask`.

In [ ]:
from transformers import Wav2Vec2FeatureExtractor

feature_extractor = Wav2Vec2FeatureExtractor(feature_size=1, sampling_rate=16000, padding_value=0.0, do_normalize=True, return_attention_mask=True)


* For improved user-friendliness, the feature extractor and tokenizer are *wrapped* into a single `Wav2Vec2Processor` class so that one only needs a `model` and `processor` object.

In [ ]:
from transformers import Wav2Vec2Processor

processor = Wav2Vec2Processor(feature_extractor=feature_extractor, tokenizer=tokenizer)

In [ ]:
print(processor)

## Data Preparation

In [ ]:
common_voice_train[0]["path"]

* **Note**: XLS-R expects the input in the format of a 1-dimensional array of 16 kHz. This means that the audio file has to be loaded and resampled.

In [ ]:
# checking the original sampling rate
common_voice_train[0]["audio"]

## Using 'Audio' function from datasets we can resample the audio files

In [ ]:
from datasets import Audio

In [ ]:
common_voice_train = common_voice_train.cast_column("audio", Audio(sampling_rate=16_000))
common_voice_test = common_voice_test.cast_column("audio", Audio(sampling_rate=16_000))

In [ ]:
common_voice_train[0]["audio"]



**Note**: *You can click the following cell a couple of times to listen to different speech samples.*

In [ ]:
import IPython.display as ipd
import numpy as np
import random

rand_int = random.randint(0, len(common_voice_train)-1)

print(common_voice_train[rand_int]["sentence"])
ipd.Audio(data=common_voice_train[rand_int]["audio"]["array"], autoplay=True, rate=16000)



Let's do a final check that the data is correctly prepared, by printing the shape of the speech input, its transcription, and the corresponding sampling rate.

**Note**: *You can click the following cell a couple of times to verify multiple samples.*

In [ ]:
rand_int = random.randint(0, len(common_voice_train)-1)

print("Target text:", common_voice_train[rand_int]["sentence"])
print("Input array shape:", common_voice_train[rand_int]["audio"]["array"].shape)
print("Sampling rate:", common_voice_train[rand_int]["audio"]["sampling_rate"])

* Finally, we can leverage `Wav2Vec2Processor` to process the data to the format expected by `Wav2Vec2ForCTC` for training. To do so let's make use of Dataset's 'map' function.

* First, we load and resample the audio data, simply by calling `batch["audio"]`.
* Second, we extract the `input_values` from the loaded audio file. In our case, the `Wav2Vec2Processor` only normalizes the data.

* Third, we encode the transcriptions to label ids.

* **Note**: This mapping function is a good example of how the `Wav2Vec2Processor` class should be used. In "normal" context, calling `processor(...)` is redirected to `Wav2Vec2FeatureExtractor`'s call method. When wrapping the processor into the `as_target_processor` context, however, the same method is redirected to `Wav2Vec2CTCTokenizer`'s call method.

## Data preparation
* For models like Wav2Vec XLS-R, which process audio in chunks or frames, knowing the input length helps in aligning frames correctly and padding them appropriately during batch training.
* In tasks like speech recognition, the input audio is often aligned with corresponding text labels or transcriptions. The input length helps maintain alignment between audio frames and corresponding tokens during training and decoding.

In [ ]:
def prepare_dataset(batch):
    # Extract audio data from the batch
    audio = batch["audio"]

    # Process audio data using the processor to obtain input values
    processed_input = processor(audio["array"], sampling_rate=audio["sampling_rate"])

    # Assign processed input values to the batch with un-batching (taking the first item of the batched output)
    batch["input_values"] = processed_input.input_values[0]

    # Calculate the input length (number of elements in input_values)
    batch["input_length"] = len(batch["input_values"])

    # Tokenize the sentence in the batch using the target processor to obtain labels
    with processor.as_target_processor():
        batch["labels"] = processor(batch["sentence"]).input_ids

    return batch


In [ ]:
common_voice_train = common_voice_train.map(prepare_dataset, remove_columns=common_voice_train.column_names)
common_voice_test = common_voice_test.map(prepare_dataset, remove_columns=common_voice_test.column_names)

In [ ]:
print(common_voice_train[0])

* **Note**: Currently `datasets` make use of `torchaudio` and `librosa` for audio loading and resampling.

## Training Setup

### Set the data collator to handle and pass the data to the model


In [ ]:
import torch

from dataclasses import dataclass, field
from typing import Any, Dict, List, Optional, Union

@dataclass
class DataCollatorCTCWithPadding:
    """
    Data collator that will dynamically pad the inputs received.
    Args:
        processor (:class:`~transformers.Wav2Vec2Processor`)
            The processor used for proccessing the data.
        padding (:obj:`bool`, :obj:`str` or :class:`~transformers.tokenization_utils_base.PaddingStrategy`, `optional`, defaults to :obj:`True`):
            Select a strategy to pad the returned sequences (according to the model's padding side and padding index)
            among:
            * :obj:`True` or :obj:`'longest'`: Pad to the longest sequence in the batch (or no padding if only a single
              sequence if provided).
            * :obj:`'max_length'`: Pad to a maximum length specified with the argument :obj:`max_length` or to the
              maximum acceptable input length for the model if that argument is not provided.
            * :obj:`False` or :obj:`'do_not_pad'` (default): No padding (i.e., can output a batch with sequences of
              different lengths).
    """

    processor: Wav2Vec2Processor
    padding: Union[bool, str] = True

    def __call__(self, features: List[Dict[str, Union[List[int], torch.Tensor]]]) -> Dict[str, torch.Tensor]:
        # split inputs and labels since they have to be of different lenghts and need
        # different padding methods
        input_features = [{"input_values": feature["input_values"]} for feature in features]
        label_features = [{"input_ids": feature["labels"]} for feature in features]

        batch = self.processor.pad(
            input_features,
            padding=self.padding,
            return_tensors="pt",
        )
        with self.processor.as_target_processor():
            labels_batch = self.processor.pad(
                label_features,
                padding=self.padding,
                return_tensors="pt",
            )

        # replace padding with -100 to ignore loss correctly
        labels = labels_batch["input_ids"].masked_fill(labels_batch.attention_mask.ne(1), -100)

        batch["labels"] = labels

        return batch

In [ ]:
data_collator = DataCollatorCTCWithPadding(processor=processor, padding=True)

In [ ]:
print(data_collator)

## Defining the evaluation metrics

In [ ]:
from datasets import load_metric

wer_metric = load_metric("wer")

## Function for evalaution

In [ ]:
def compute_metrics(pred):
    pred_logits = pred.predictions
    pred_ids = np.argmax(pred_logits, axis=-1)

    pred.label_ids[pred.label_ids == -100] = processor.tokenizer.pad_token_id

    pred_str = processor.batch_decode(pred_ids)
    # we do not want to group tokens when computing the metrics
    label_str = processor.batch_decode(pred.label_ids, group_tokens=False)

    wer = wer_metric.compute(predictions=pred_str, references=label_str)

    return {"wer": wer}

## Loading the pre-trained model
* Here we use 'xls-r-300m' variant.

In [ ]:
from transformers import Wav2Vec2ForCTC

model = Wav2Vec2ForCTC.from_pretrained(
    "facebook/wav2vec2-xls-r-300m",  # Load pre-trained model weights from Hugging Face model hub
    attention_dropout=0.0,  # Dropout probability for attention layers (no dropout in this case)
    hidden_dropout=0.0,  # Dropout probability for hidden layers (no dropout in this case)
    feat_proj_dropout=0.0,  # Dropout probability for feature projection layers (no dropout in this case)
    mask_time_prob=0.05,  # Probability of applying time masking during training (5% of frames are masked)
    layerdrop=0.0,    # LayerDrop probability (no layer dropping in this case)
    ctc_loss_reduction="mean",  # Reduction method for CTC loss during training
    pad_token_id=processor.tokenizer.pad_token_id,  # ID of padding token in tokenizer's vocabulary
    vocab_size=len(processor.tokenizer),  # Vocabulary size based on tokenizer's vocabulary
)


In [ ]:
model.freeze_feature_extractor()

## Defining training parameters

#### **Note**:The parameters used in the following code are for tutorial purposes only. Modify the standard parameters, such as epochs, saving frequency, evaluation intervals, etc., according to the specific requirements of your task.







In [ ]:
from transformers import TrainingArguments

training_args = TrainingArguments(
  output_dir = 'output',
  group_by_length=True,
  per_device_train_batch_size=16,
  gradient_accumulation_steps=2,
  evaluation_strategy="steps",
  num_train_epochs=2,
  gradient_checkpointing=True,
  fp16=True,    # mixed precision training (16-bit precision) to speed up training and save memory
  save_steps=50,
  eval_steps=50,
  logging_steps=20,
  learning_rate=3e-4,
  warmup_steps=20,
  save_total_limit=2,
  push_to_hub=False,
)

In [ ]:
from transformers import Trainer

trainer = Trainer(
    model=model,
    data_collator=data_collator,
    args=training_args,
    compute_metrics=compute_metrics,
    train_dataset=common_voice_train,
    eval_dataset=common_voice_test,
    tokenizer=processor.feature_extractor,
)

### Training

In [ ]:
trainer.train()

In [ ]:
# pushing the traiend model to the hub
# trainer.push_to_hub()

You can now share this model with all your friends, family, favorite pets: they can all load it with the identifier "your-username/the-name-you-picked" so for instance:

### Evaluation

As a final check, let's load the model and verify that it indeed has learned to transcribe Hindi speech.

Let's first load the pretrained checkpoint.

In [ ]:
# model = Wav2Vec2ForCTC.from_pretrained('repo_name').to("cuda")
# processor = Wav2Vec2Processor.from_pretrained('repo_name')


Now, we will just take the first example of the test set, run it through the model and take the `argmax(...)` of the logits to retrieve the predicted token ids.

In [ ]:
# input_dict = processor(common_voice_test[0]["input_values"], return_tensors="pt", padding=True)

# logits = model(input_dict.input_values.to("cuda")).logits

# pred_ids = torch.argmax(logits, dim=-1)[0]

We adapted `common_voice_test` quite a bit so that the dataset instance does not contain the original sentence label anymore. Thus, we re-use the original dataset to get the label of the first example.

In [ ]:
# common_voice_test_transcription = load_dataset("common_voice", "hi", data_dir=" ", split="test")


Finally, we can decode the example.

In [ ]:
# print("Prediction:")
# print(processor.decode(pred_ids))

# print("\nReference:")
# print(common_voice_test_transcription[0]["sentence"].lower())